# Sistema de recomendación: filtrado colaborativo item-item

Este notebook utiliza `pedidos`, `usuarios` y `productos`. Construye una matriz usuario-producto a partir de pedidos entregados y calcula similitud coseno entre productos.

## 1. Dependencias
Instalación opcional: `pip install pymongo python-dotenv pandas scikit-learn matplotlib seaborn`.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pymongo import MongoClient
from sklearn.metrics.pairwise import cosine_similarity

env_path = next((p for p in [Path.cwd() / '.env', Path.cwd().parent / '.env'] if p.exists()), None)
load_dotenv(env_path)
MONGO_URI = os.getenv('MONGO_URI')
if not MONGO_URI:
    raise RuntimeError('No se encontró MONGO_URI. Abre Jupyter desde pryBinaBack o carga el archivo .env.')

## 2. Extraer interacciones de pedidos entregados

In [ ]:
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000)
db = client.get_default_database()
pedidos = list(db.pedidos.find({'estado': 'Entregado'}, {'usuario': 1, 'productos': 1, 'createdAt': 1}))
filas = []
for pedido in pedidos:
    for item in pedido.get('productos', []):
        filas.append({
            'pedido_id': str(pedido['_id']),
            'usuario': str(pedido.get('usuario', '')),
            'producto': str(item.get('producto', '')),
            'cantidad': item.get('cantidad', 0),
            'fecha': pedido.get('createdAt')
        })
interacciones = pd.DataFrame(filas)
print('Pedidos entregados:', len(pedidos))
print('Usuarios:', interacciones['usuario'].nunique())
print('Productos:', interacciones['producto'].nunique())
display(interacciones.head())

## 3. Matriz usuario-producto
Se utiliza la cantidad total comprada como intensidad de interacción. También puede cambiarse a valores binarios con `(matriz > 0).astype(int)`.

In [ ]:
matriz = interacciones.pivot_table(index='usuario', columns='producto', values='cantidad', aggfunc='sum', fill_value=0)
print('Dimensiones usuario-producto:', matriz.shape)
display(matriz.iloc[:5, :8])
if matriz.shape[0] < 20:
    print('ADVERTENCIA: hay pocos usuarios. Para mejores resultados genera más perfiles con patrones de compra diferentes.')

In [ ]:
similitud = cosine_similarity(matriz.T)
similitud_items = pd.DataFrame(similitud, index=matriz.columns, columns=matriz.columns)
similitud_items.iloc[:5, :5]

## 4. Catálogo para mostrar nombres, marca y familia

In [ ]:
productos = list(db.productos.find({}, {'nombre': 1, 'marca': 1, 'familia': 1, 'precioNormal': 1, 'stock': 1, 'activo': 1}))
catalogo = pd.DataFrame([{
    'producto': str(p['_id']), 'nombre': p.get('nombre'), 'marca': str(p.get('marca', '')),
    'familia': str(p.get('familia', '')), 'precio': p.get('precioNormal', 0),
    'stock': p.get('stock', 0), 'activo': p.get('activo', True)
} for p in productos]).set_index('producto')
display(catalogo.head())

## 5. Productos similares a un producto

In [ ]:
def productos_similares(producto_id, top_n=6):
    if producto_id not in similitud_items.index:
        return pd.DataFrame(columns=['nombre', 'precio', 'similitud'])
    scores = similitud_items.loc[producto_id].drop(producto_id).sort_values(ascending=False)
    resultado = scores.rename('similitud').to_frame().join(catalogo, how='left')
    resultado = resultado[(resultado['activo'] != False) & (resultado['stock'] > 0)]
    return resultado[['nombre', 'marca', 'familia', 'precio', 'similitud']].head(top_n)

producto_ejemplo = matriz.columns[0]
print('Producto base:', catalogo.loc[producto_ejemplo, 'nombre'] if producto_ejemplo in catalogo.index else producto_ejemplo)
display(productos_similares(producto_ejemplo))

## 6. Recomendaciones para un carrito mixto
La puntuación acumula la similitud respecto de todos los productos del carrito. Se excluyen los productos que ya están presentes.

In [ ]:
def recomendar_carrito(producto_ids, top_n=6):
    semillas = [p for p in dict.fromkeys(producto_ids) if p in similitud_items.index]
    if not semillas:
        return pd.DataFrame(columns=['nombre', 'precio', 'puntuacion'])
    puntuacion = similitud_items.loc[semillas].sum(axis=0)
    puntuacion = puntuacion.drop(labels=semillas, errors='ignore').sort_values(ascending=False)
    resultado = puntuacion.rename('puntuacion').to_frame().join(catalogo, how='left')
    resultado = resultado[(resultado['activo'] != False) & (resultado['stock'] > 0)]
    return resultado[['nombre', 'marca', 'familia', 'precio', 'puntuacion']].head(top_n)

carrito_ejemplo = list(matriz.columns[:3])
print('Carrito:', catalogo.reindex(carrito_ejemplo)['nombre'].dropna().tolist())
display(recomendar_carrito(carrito_ejemplo))

## 7. Visualizar similitudes de una muestra

In [ ]:
muestra = similitud_items.columns[:15]
plt.figure(figsize=(12, 8))
sns.heatmap(similitud_items.loc[muestra, muestra], cmap='YlGnBu', vmin=0, vmax=1)
plt.title('Similitud coseno entre productos (muestra)')
plt.tight_layout(); plt.show()
client.close()